# Fase 4 - Regresión: Predicción de Tarifa de Taxi

**Pregunta de negocio:** ¿Podemos predecir la tarifa de un viaje?

## Objetivos de este notebook

1. Cargar el dataset con features preprocesados (output del notebook 01)
2. Establecer un baseline: predecir la tarifa media
3. Entrenar y evaluar modelos progresivamente más complejos:
   - Regresión Lineal
   - Ridge Regression (con tuning de alpha)
   - Lasso Regression (con efecto de selección de features)
   - Random Forest Regressor
   - XGBoost Regressor
4. Comparar modelos con métricas estándar (R², MAE, RMSE)
5. Analizar residuos del mejor modelo
6. Interpretar feature importance
7. Guardar el mejor modelo para producción

## ¿Por qué fare_amount es un buen primer problema de regresión?

La tarifa de un taxi está determinada en gran medida por una fórmula conocida:
tarifa base + costo por milla + costo por minuto de espera. Esto significa que
un modelo que conozca la distancia y duración debería predecir la tarifa con
alta precisión. Es un excelente problema para aprender los fundamentos de la
regresión antes de abordar targets más complejos.

## 1. Configuración e importaciones

In [ ]:
# Conexión a BigQuery (disponible si necesitamos recargar datos)
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import joblib

# Scikit-learn: modelos y métricas
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# XGBoost
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 2. Carga del dataset preprocesado

Cargamos el parquet generado en el notebook 01_feature_engineering.
Si no existe, recreamos los features directamente desde BigQuery.

In [ ]:
data_path = '../../../data/processed/nyc_taxi/features_engineered.parquet'

if os.path.exists(data_path):
    df = pd.read_parquet(data_path)
    print(f"Dataset cargado desde parquet: {df.shape}")
else:
    print("Archivo parquet no encontrado. Ejecuta primero 01_feature_engineering.ipynb")
    print("Cargando datos directamente desde BigQuery como alternativa...")
    
    query = """
    SELECT
        trip_distance,
        TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) / 60.0 AS duration_min,
        CASE
            WHEN TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) > 0 AND trip_distance > 0
            THEN trip_distance / (TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) / 3600.0)
            ELSE NULL
        END AS speed_mph,
        passenger_count,
        EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour,
        EXTRACT(DAYOFWEEK FROM pickup_datetime) - 1 AS pickup_day_of_week,
        EXTRACT(MONTH FROM pickup_datetime) AS month,
        fare_amount,
        tip_amount,
        CASE WHEN fare_amount > 0 THEN (tip_amount / fare_amount) * 100 ELSE NULL END AS tip_pct,
        payment_type
    FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
    WHERE fare_amount BETWEEN 2.5 AND 200
        AND trip_distance BETWEEN 0.1 AND 50
        AND passenger_count BETWEEN 1 AND 6
        AND pickup_location_id IS NOT NULL
        AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) BETWEEN 60 AND 7200
        AND tip_amount >= 0
    ORDER BY RAND()
    LIMIT 200000
    """
    df = bq.query_to_df(query)
    df = df.dropna()
    # Crear features mínimos
    df['is_weekend'] = (df['pickup_day_of_week'] >= 5).astype(int)
    df['is_rush_hour'] = df['pickup_hour'].apply(lambda h: 1 if (7<=h<=9) or (17<=h<=19) else 0)
    df['is_night'] = df['pickup_hour'].apply(lambda h: 1 if h >= 20 or h <= 6 else 0)
    df['fare_per_mile'] = np.where(df['trip_distance'] > 0, df['fare_amount'] / df['trip_distance'], 0)
    df['fare_per_minute'] = np.where(df['duration_min'] > 0, df['fare_amount'] / df['duration_min'], 0)
    print(f"Dataset cargado desde BigQuery: {df.shape}")

df.head()

In [ ]:
# Separar features y target
target_col = 'fare_amount'
exclude_cols = ['fare_amount', 'tip_amount', 'tip_pct']  # Targets no son features

feature_cols = [col for col in df.columns if col not in exclude_cols]
X = df[feature_cols].copy()
y = df[target_col].copy()

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nFeatures: {list(X.columns)}")
print(f"\nTarget (fare_amount):")
print(f"  Media:   ${y.mean():.2f}")
print(f"  Mediana: ${y.median():.2f}")
print(f"  Std:     ${y.std():.2f}")
print(f"  Rango:   [${y.min():.2f}, ${y.max():.2f}]")

## 3. Train/Test Split

Dividimos 80% para entrenamiento y 20% para prueba. Usamos `random_state=42`
para reproducibilidad.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Entrenamiento: X={X_train.shape}, y={y_train.shape}")
print(f"Prueba:        X={X_test.shape}, y={y_test.shape}")
print(f"\nDistribución del target en train: media=${y_train.mean():.2f}, std=${y_train.std():.2f}")
print(f"Distribución del target en test:  media=${y_test.mean():.2f}, std=${y_test.std():.2f}")

# Escalar features para modelos lineales
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Baseline: Predecir la tarifa media

El baseline más simple: predecir siempre la tarifa promedio del set de entrenamiento.
Cualquier modelo útil debe superar este baseline.

In [ ]:
# Función auxiliar para calcular y mostrar métricas
def evaluate_model(y_true, y_pred, model_name):
    """Calcula R², MAE y RMSE para un modelo."""
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"  R²:   {r2:.4f}")
    print(f"  MAE:  ${mae:.2f}")
    print(f"  RMSE: ${rmse:.2f}")
    return {'Modelo': model_name, 'R²': r2, 'MAE': mae, 'RMSE': rmse}

# Diccionario para almacenar resultados
results = []

# Baseline: predecir la media
y_pred_baseline = np.full_like(y_test, y_train.mean())

print("=" * 50)
print("BASELINE: Predecir tarifa media")
print(f"Predicción constante: ${y_train.mean():.2f}")
print("=" * 50)
result = evaluate_model(y_test, y_pred_baseline, 'Baseline (media)')
result['Tiempo (s)'] = 0.0
results.append(result)

## 5. Regresión Lineal

El modelo más básico de regresión supervisada. Encuentra los coeficientes que
minimizan la suma de errores cuadráticos. No tiene hiperparámetros que ajustar.

In [ ]:
print("=" * 50)
print("REGRESIÓN LINEAL")
print("=" * 50)

t0 = time.time()
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
train_time = time.time() - t0

result = evaluate_model(y_test, y_pred_lr, 'Regresión Lineal')
result['Tiempo (s)'] = round(train_time, 3)
results.append(result)

# Coeficientes
coefs = pd.Series(lr.coef_, index=feature_cols).sort_values(key=abs, ascending=False)
print(f"\nIntercepto: {lr.intercept_:.4f}")
print(f"\nTop 5 coeficientes (mayor magnitud):")
for feat, coef in coefs.head(5).items():
    print(f"  {feat:25s}: {coef:+.4f}")

## 6. Ridge Regression (L2)

Ridge agrega una penalización L2 que previene coeficientes muy grandes.
El hiperparámetro **alpha** controla la fuerza de la regularización.
Usamos validación cruzada para encontrar el mejor alpha.

In [ ]:
print("=" * 50)
print("RIDGE REGRESSION - Búsqueda de alpha")
print("=" * 50)

alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
ridge_scores = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    cv_scores = cross_val_score(ridge, X_train_scaled, y_train, cv=5, scoring='r2')
    ridge_scores.append({
        'alpha': alpha,
        'mean_r2': cv_scores.mean(),
        'std_r2': cv_scores.std()
    })
    print(f"  alpha={alpha:8.3f} | R² = {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Mejor alpha
ridge_df = pd.DataFrame(ridge_scores)
best_alpha = ridge_df.loc[ridge_df['mean_r2'].idxmax(), 'alpha']
print(f"\nMejor alpha: {best_alpha}")

In [ ]:
# Entrenar Ridge con mejor alpha
t0 = time.time()
ridge_best = Ridge(alpha=best_alpha)
ridge_best.fit(X_train_scaled, y_train)
y_pred_ridge = ridge_best.predict(X_test_scaled)
train_time = time.time() - t0

print(f"Ridge Regression (alpha={best_alpha}):")
result = evaluate_model(y_test, y_pred_ridge, f'Ridge (alpha={best_alpha})')
result['Tiempo (s)'] = round(train_time, 3)
results.append(result)

# Visualizar efecto de alpha en los coeficientes
fig, ax = plt.subplots(figsize=(10, 6))
ridge_df.plot(x='alpha', y='mean_r2', marker='o', ax=ax, color='#2196F3', legend=False)
ax.fill_between(ridge_df['alpha'],
                ridge_df['mean_r2'] - ridge_df['std_r2'],
                ridge_df['mean_r2'] + ridge_df['std_r2'],
                alpha=0.2, color='#2196F3')
ax.set_xscale('log')
ax.set_title('Ridge: R² vs Alpha (Validación Cruzada)', fontweight='bold')
ax.set_xlabel('Alpha (escala logarítmica)')
ax.set_ylabel('R² (CV 5-fold)')
ax.axvline(best_alpha, color='red', linestyle='--', label=f'Mejor alpha={best_alpha}')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Lasso Regression (L1)

Lasso aplica una penalización L1 que puede llevar coeficientes exactamente a cero,
actuando así como un método de **selección de features**. Esto es especialmente
útil cuando tenemos muchos features y queremos identificar los más relevantes.

In [ ]:
print("=" * 50)
print("LASSO REGRESSION - Búsqueda de alpha")
print("=" * 50)

alphas_lasso = [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]
lasso_scores = []

for alpha in alphas_lasso:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    cv_scores = cross_val_score(lasso, X_train_scaled, y_train, cv=5, scoring='r2')
    
    # Contar features con coeficiente no-cero
    lasso_temp = Lasso(alpha=alpha, max_iter=10000)
    lasso_temp.fit(X_train_scaled, y_train)
    n_nonzero = np.sum(lasso_temp.coef_ != 0)
    
    lasso_scores.append({
        'alpha': alpha,
        'mean_r2': cv_scores.mean(),
        'std_r2': cv_scores.std(),
        'n_features': n_nonzero
    })
    print(f"  alpha={alpha:8.4f} | R²={cv_scores.mean():.4f} | Features activos: {n_nonzero}/{len(feature_cols)}")

lasso_df = pd.DataFrame(lasso_scores)
best_alpha_lasso = lasso_df.loc[lasso_df['mean_r2'].idxmax(), 'alpha']
print(f"\nMejor alpha: {best_alpha_lasso}")

In [ ]:
# Entrenar Lasso con mejor alpha
t0 = time.time()
lasso_best = Lasso(alpha=best_alpha_lasso, max_iter=10000)
lasso_best.fit(X_train_scaled, y_train)
y_pred_lasso = lasso_best.predict(X_test_scaled)
train_time = time.time() - t0

print(f"Lasso Regression (alpha={best_alpha_lasso}):")
result = evaluate_model(y_test, y_pred_lasso, f'Lasso (alpha={best_alpha_lasso})')
result['Tiempo (s)'] = round(train_time, 3)
results.append(result)

# Mostrar qué features fueron eliminados (coeficiente = 0)
lasso_coefs = pd.Series(lasso_best.coef_, index=feature_cols)
zero_coefs = lasso_coefs[lasso_coefs == 0]
nonzero_coefs = lasso_coefs[lasso_coefs != 0].sort_values(key=abs, ascending=False)

print(f"\nFeatures eliminados por Lasso (coeficiente = 0): {len(zero_coefs)}")
for feat in zero_coefs.index:
    print(f"  - {feat}")

print(f"\nFeatures retenidos ({len(nonzero_coefs)}):")
for feat, coef in nonzero_coefs.items():
    print(f"  {feat:25s}: {coef:+.4f}")

## 8. Random Forest Regressor

Random Forest es un ensemble de árboles de decisión. Cada árbol se entrena con
un subconjunto aleatorio de datos y features. No requiere escalado de variables
y captura relaciones no lineales.

In [ ]:
print("=" * 50)
print("RANDOM FOREST - Default")
print("=" * 50)

# Primero con parámetros por defecto
t0 = time.time()
rf_default = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_default.fit(X_train, y_train)  # Sin escalado
y_pred_rf_default = rf_default.predict(X_test)
train_time_default = time.time() - t0

print("Parámetros por defecto (n_estimators=100):")
result_rf_default = evaluate_model(y_test, y_pred_rf_default, 'Random Forest (default)')
result_rf_default['Tiempo (s)'] = round(train_time_default, 3)
print(f"  Tiempo: {train_time_default:.1f}s")

In [ ]:
# Tuning de Random Forest
print("\n" + "=" * 50)
print("RANDOM FOREST - Tuning")
print("=" * 50)

rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_params,
    cv=3,
    scoring='r2',
    verbose=1,
    n_jobs=-1
)

t0 = time.time()
rf_grid.fit(X_train, y_train)
train_time_rf = time.time() - t0

print(f"\nMejores parámetros: {rf_grid.best_params_}")
print(f"Mejor R² (CV): {rf_grid.best_score_:.4f}")

y_pred_rf = rf_grid.best_estimator_.predict(X_test)
print("\nEvaluación en test:")
result = evaluate_model(y_test, y_pred_rf, 'Random Forest (tuned)')
result['Tiempo (s)'] = round(train_time_rf, 3)
results.append(result)

## 9. XGBoost Regressor

XGBoost (eXtreme Gradient Boosting) construye árboles secuencialmente, donde
cada nuevo árbol intenta corregir los errores del anterior. Suele ofrecer
el mejor rendimiento entre los modelos clásicos de ML.

In [ ]:
print("=" * 50)
print("XGBOOST - Default")
print("=" * 50)

t0 = time.time()
xgb_default = XGBRegressor(
    n_estimators=100, random_state=42, n_jobs=-1,
    verbosity=0
)
xgb_default.fit(X_train, y_train)
y_pred_xgb_default = xgb_default.predict(X_test)
train_time_xgb_def = time.time() - t0

print("Parámetros por defecto:")
evaluate_model(y_test, y_pred_xgb_default, 'XGBoost (default)')
print(f"  Tiempo: {train_time_xgb_def:.1f}s")

In [ ]:
# Tuning de XGBoost con GridSearchCV
print("\n" + "=" * 50)
print("XGBOOST - GridSearchCV Tuning")
print("=" * 50)

xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
}

xgb_grid = GridSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    xgb_params,
    cv=3,
    scoring='r2',
    verbose=1,
    n_jobs=-1
)

t0 = time.time()
xgb_grid.fit(X_train, y_train)
train_time_xgb = time.time() - t0

print(f"\nMejores parámetros: {xgb_grid.best_params_}")
print(f"Mejor R² (CV): {xgb_grid.best_score_:.4f}")

y_pred_xgb = xgb_grid.best_estimator_.predict(X_test)
print("\nEvaluación en test:")
result = evaluate_model(y_test, y_pred_xgb, 'XGBoost (tuned)')
result['Tiempo (s)'] = round(train_time_xgb, 3)
results.append(result)

## 10. Comparación de modelos

Compilamos las métricas de todos los modelos para una comparación clara.

In [ ]:
# Tabla comparativa
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('R²', ascending=False).reset_index(drop=True)

print("=" * 70)
print("COMPARACIÓN DE MODELOS - Predicción de Tarifa")
print("=" * 70)
print(results_df.to_string(index=False))

In [ ]:
# Visualización de comparación
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = ['R²', 'MAE', 'RMSE']
colors_map = {
    'R²': '#4CAF50',
    'MAE': '#FF9800',
    'RMSE': '#F44336'
}

for ax, metric in zip(axes, metrics):
    bars = ax.barh(results_df['Modelo'], results_df[metric], color=colors_map[metric], alpha=0.8)
    ax.set_title(f'Comparación: {metric}', fontweight='bold')
    ax.set_xlabel(metric)
    
    # Agregar valores en las barras
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
                f' {val:.4f}' if metric == 'R²' else f' ${val:.2f}',
                va='center', fontsize=9)

plt.suptitle('Comparación de Modelos de Regresión - Predicción de Tarifa',
             fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 11. Análisis de residuos

Los residuos (error = valor real - valor predicho) nos ayudan a entender
dónde y cómo falla el modelo. Un modelo ideal tendría residuos aleatorios,
centrados en cero y con distribución normal.

In [ ]:
# Usamos el mejor modelo (XGBoost tuned o Random Forest tuned)
best_model_name = results_df.iloc[0]['Modelo']
print(f"Análisis de residuos para: {best_model_name}")

# Determinar las predicciones del mejor modelo
if 'XGBoost' in best_model_name:
    y_pred_best = y_pred_xgb
    best_model = xgb_grid.best_estimator_
elif 'Random Forest' in best_model_name and 'tuned' in best_model_name:
    y_pred_best = y_pred_rf
    best_model = rf_grid.best_estimator_
else:
    y_pred_best = y_pred_lr
    best_model = lr

residuals = y_test.values - y_pred_best

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Residuos vs Predichos
axes[0, 0].scatter(y_pred_best, residuals, alpha=0.1, s=5, color='#2196F3')
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_title('Residuos vs Valores Predichos', fontweight='bold')
axes[0, 0].set_xlabel('Tarifa Predicha (USD)')
axes[0, 0].set_ylabel('Residuo (USD)')
axes[0, 0].set_ylim(-30, 30)

# 2. Distribución de residuos
axes[0, 1].hist(residuals, bins=100, color='#4CAF50', alpha=0.7, edgecolor='white', density=True)
from scipy import stats
x_range = np.linspace(residuals.min(), residuals.max(), 100)
axes[0, 1].plot(x_range, stats.norm.pdf(x_range, residuals.mean(), residuals.std()),
                'r-', linewidth=2, label='Normal teórica')
axes[0, 1].set_title('Distribución de Residuos', fontweight='bold')
axes[0, 1].set_xlabel('Residuo (USD)')
axes[0, 1].set_ylabel('Densidad')
axes[0, 1].set_xlim(-30, 30)
axes[0, 1].legend()

# 3. QQ plot
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('QQ Plot de Residuos', fontweight='bold')
axes[1, 0].get_lines()[0].set_markersize(2)

# 4. Predichos vs Reales
axes[1, 1].scatter(y_test, y_pred_best, alpha=0.1, s=5, color='#FF9800')
max_val = max(y_test.max(), y_pred_best.max())
axes[1, 1].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Predicción perfecta')
axes[1, 1].set_title('Valores Reales vs Predichos', fontweight='bold')
axes[1, 1].set_xlabel('Tarifa Real (USD)')
axes[1, 1].set_ylabel('Tarifa Predicha (USD)')
axes[1, 1].set_xlim(0, 80)
axes[1, 1].set_ylim(0, 80)
axes[1, 1].legend()

plt.suptitle(f'Análisis de Residuos - {best_model_name}',
             fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nEstadísticas de residuos:")
print(f"  Media:   {residuals.mean():.4f}")
print(f"  Std:     {residuals.std():.4f}")
print(f"  Mediana: {np.median(residuals):.4f}")

In [ ]:
# Residuos por zona (si la columna existe)
zone_cols = [col for col in X_test.columns if col.startswith('zone_')]

if zone_cols:
    # Reconstruir zona desde one-hot
    test_zones = X_test[zone_cols].copy()
    test_zones['zone'] = 'Referencia'  # Categoría dropeada
    for col in zone_cols:
        zone_name = col.replace('zone_', '')
        test_zones.loc[test_zones[col] == 1, 'zone'] = zone_name
    
    residuals_by_zone = pd.DataFrame({
        'residual': residuals,
        'zone': test_zones['zone'].values
    })
    
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(data=residuals_by_zone, x='zone', y='residual', 
                palette='Set2', showfliers=False, ax=ax)
    ax.axhline(y=0, color='red', linestyle='--')
    ax.set_title('Distribución de Residuos por Zona', fontweight='bold')
    ax.set_xlabel('Zona de Recogida')
    ax.set_ylabel('Residuo (USD)')
    plt.tight_layout()
    plt.show()
    
    print("\nMAE por zona:")
    for zone in residuals_by_zone['zone'].unique():
        mask = residuals_by_zone['zone'] == zone
        mae_zone = np.abs(residuals_by_zone.loc[mask, 'residual']).mean()
        count = mask.sum()
        print(f"  {zone:15s}: MAE=${mae_zone:.2f} (n={count:,})")
else:
    print("No se encontraron columnas de zona en los features.")

## 12. Feature Importance del mejor modelo

La importancia de features nos dice qué variables contribuyen más a la predicción.
Para modelos basados en árboles, se basa en cuánto reduce cada feature la impureza
(error) al hacer splits.

In [ ]:
# Feature importance del mejor modelo
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(
        best_model.feature_importances_,
        index=feature_cols
    ).sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    importances.plot(kind='barh', ax=ax, color='#2196F3')
    ax.set_title(f'Feature Importance - {best_model_name}', fontweight='bold')
    ax.set_xlabel('Importancia')
    plt.tight_layout()
    plt.show()
    
    print("\nTop 10 features más importantes:")
    for i, (feat, imp) in enumerate(importances.sort_values(ascending=False).head(10).items(), 1):
        print(f"  {i:2d}. {feat:25s}: {imp:.4f} ({imp*100:.1f}%)")
else:
    print("El mejor modelo no tiene atributo feature_importances_.")
    print("Usando coeficientes del modelo lineal en su lugar.")
    coefs_abs = pd.Series(np.abs(best_model.coef_), index=feature_cols).sort_values(ascending=True)
    coefs_abs.plot(kind='barh', figsize=(10, 8), color='#2196F3')
    plt.title('Magnitud de Coeficientes (modelo lineal)', fontweight='bold')
    plt.xlabel('|Coeficiente|')
    plt.tight_layout()
    plt.show()

## 13. Insight clave: la distancia domina

Como era de esperarse, **trip_distance** y **duration_min** dominan la predicción
de la tarifa. Esto tiene sentido porque la tarifa del taxi se calcula principalmente
como una función de la distancia y el tiempo.

El alto R² (~0.95) confirma que la tarifa es altamente predecible. Esto no es
sorprendente dado que sigue una fórmula determinista con poco ruido.

**Implicaciones:**
- Para una app de estimación de tarifa, solo necesitaríamos distancia y duración estimada
- Los features temporales y geográficos capturan las variaciones residuales (recargos nocturnos, tarifas de aeropuerto)
- Un modelo lineal simple ya logra un rendimiento muy alto en este problema

## 14. Guardar el mejor modelo

In [ ]:
# Guardar el mejor modelo
model_dir = '../../../models/nyc_taxi'
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, 'fare_model.joblib')

# Guardar modelo y metadatos
model_artifacts = {
    'model': best_model,
    'scaler': scaler,
    'feature_names': feature_cols,
    'model_name': best_model_name,
    'metrics': {
        'R2': float(results_df.iloc[0]['R²']),
        'MAE': float(results_df.iloc[0]['MAE']),
        'RMSE': float(results_df.iloc[0]['RMSE'])
    }
}

joblib.dump(model_artifacts, model_path)

print(f"Modelo guardado en: {model_path}")
print(f"Modelo: {best_model_name}")
print(f"Métricas:")
print(f"  R²:   {model_artifacts['metrics']['R2']:.4f}")
print(f"  MAE:  ${model_artifacts['metrics']['MAE']:.2f}")
print(f"  RMSE: ${model_artifacts['metrics']['RMSE']:.2f}")
print(f"\nTamaño del archivo: {os.path.getsize(model_path) / 1024 / 1024:.1f} MB")

# Verificar carga
loaded = joblib.load(model_path)
print(f"\nVerificación: modelo cargado correctamente ({loaded['model_name']})")

## Conclusiones

1. **La tarifa es altamente predecible**: R² ~ 0.95 indica que el modelo explica
   el 95% de la varianza en la tarifa, lo cual era esperable dada la naturaleza
   formulaica de la tarifa.

2. **La distancia domina**: trip_distance es, con mucho, el feature más importante,
   seguido de duration_min. Juntos explican la mayor parte de la varianza.

3. **Modelos simples vs complejos**: La mejora de XGBoost/Random Forest sobre
   la regresión lineal es marginal en este problema, porque la relación
   tarifa-distancia es aproximadamente lineal.

4. **Lasso como selector de features**: Lasso identificó efectivamente los
   features menos relevantes llevando sus coeficientes a cero.

5. **Residuos**: Los errores más grandes ocurren en tarifas altas (aeropuertos,
   viajes largos) donde hay más variabilidad.

### Próximo paso

En el notebook 03, abordaremos un problema mucho más desafiante: predecir la propina.
A diferencia de la tarifa, la propina depende del comportamiento humano y esperamos
un R² significativamente menor.